Hypothesis(H1): Brain regions exhibit unique connectivity fingerprints at rest enabling, accurate region identification from connectivity patterns. 

In [3]:
# =============================================================================
# One-vs-Rest Brain Region Classification
# =============================================================================

try:
    import numpy as np
    import pandas as pd
    import json
    from pathlib import Path
    import warnings
    import re

    # Visualization
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.express as px

    # Machine Learning
    from sklearn.metrics import confusion_matrix

    # Configuration
    warnings.filterwarnings('ignore')
    pd.set_option('display.max_columns', None)
    pd.set_option('display.precision', 2)

    # Color Palette for Visualizations
    COLORS = {
        'baseline': '#e74c3c',      # Red
        'step1': '#f39c12',          # Orange  
        'step2': '#3498db',          # Blue
        'step3': '#2ecc71',          # Green
        'same_h_same_n': '#1A5276',
        'same_h_diff_n': '#1D8348',
        'diff_h_same_n': '#7D3C98',
        'diff_h_diff_n': '#2E4053'
    }
    print("Imports successful.")
except Exception as e:
    print(f"Import Error: {e}")

Imports successful.


In [4]:
# =============================================================================
# PATH VERIFICATION
# =============================================================================


BASE_PATH = Path('/home/sjoon/projects/brain_connectivity_classifier/data')
RESULTS_DIR = BASE_PATH / 'results'

# Template for repetitive paths
def res_p(sub, task): return RESULTS_DIR / sub / task
f_sub = 'full_connectivity_analysis'
l_sub, r_sub = 'hemisphere_analysis/left_hemisphere', 'hemisphere_analysis/right_hemisphere'
ovr, t_ovr = 'one_vs_rest', 'task_testing_one_vs_rest'

PATHS = {
    'full_cv': res_p(f_sub, ovr) / 'cv_summary.json',
    'full_task': res_p(f_sub, t_ovr) / 'task_testing_summary.json',
    'full_pred': res_p(f_sub, t_ovr) / 'task_predictions.npy',
    'full_true': res_p(f_sub, t_ovr) / 'task_true_labels.npy',
    
    'left_cv': res_p(l_sub, ovr) / 'cv_summary.json',
    'left_task': res_p(l_sub, t_ovr) / 'task_testing_summary.json',
    'left_pred': res_p(l_sub, t_ovr) / 'task_predictions.npy',
    'left_true': res_p(l_sub, t_ovr) / 'task_true_labels.npy',
    
    'right_cv': res_p(r_sub, ovr) / 'cv_summary.json',
    'right_task': res_p(r_sub, t_ovr) / 'task_testing_summary.json',
    'right_pred': res_p(r_sub, t_ovr) / 'task_predictions.npy',
    'right_true': res_p(r_sub, t_ovr) / 'task_true_labels.npy',
    
    'full_regions': BASE_PATH / 'FULL_region_info.csv',
    'lh_regions': BASE_PATH / 'LH_region_info.csv',
    'rh_regions': BASE_PATH / 'RH_region_info.csv',
    'coordinates': BASE_PATH / 'network_files' / 'Schaefer2018_200Parcels_Tian_32Parcels.csv'
}

# Condensed Verification
print("Path Verification:")
missing_paths = [n for n, p in PATHS.items() if not (print(f"  {'✓' if p.exists() else '✗'} {n}{'' if p.exists() else ' - MISSING'}") or p.exists())]

if missing_paths: print(f"\n⚠ Warning: {len(missing_paths)} path(s) missing")
else: print("\n✓ All paths verified successfully")

Path Verification:
  ✓ full_cv
  ✓ full_task
  ✓ full_pred
  ✓ full_true
  ✓ left_cv
  ✓ left_task
  ✓ left_pred
  ✓ left_true
  ✓ right_cv
  ✓ right_task
  ✓ right_pred
  ✓ right_true
  ✓ full_regions
  ✓ lh_regions
  ✓ rh_regions
  ✓ coordinates

✓ All paths verified successfully


In [5]:
# =============================================================================
# DATA LOADING
# =============================================================================
print("Loading data...\n")

# 1. Load region metadata
full_region_info = pd.read_csv(PATHS['full_regions'])
lh_region_info = pd.read_csv(PATHS['lh_regions'])
rh_region_info = pd.read_csv(PATHS['rh_regions'])
coords_df = pd.read_csv(PATHS['coordinates'])

print(f"✓ Metadata loaded: Full ({len(full_region_info)}), Left ({len(lh_region_info)}), Right ({len(rh_region_info)})")

# 2. Load performance summaries
def load_summary(path):
    with open(path, 'r') as f: return json.load(f)

s_keys = ['full_cv', 'full_task', 'left_cv', 'left_task', 'right_cv', 'right_task']
full_cv, full_task, left_cv, left_task, right_cv, right_task = [load_summary(PATHS[k]) for k in s_keys]

# 3. Load predictions and labels
p_keys = [('full_pred', 'full_true'), ('left_pred', 'left_true'), ('right_pred', 'right_true')]
preds_data = [ (np.load(PATHS[p], allow_pickle=True), np.load(PATHS[t], allow_pickle=True)) for p, t in p_keys]

(full_preds, full_true), (left_preds, left_true), (right_preds, right_true) = preds_data



print(f"\n✓ Summaries and Predictions loaded:")
print(f"  Full: {len(full_preds):,} | Left: {len(left_preds):,} | Right: {len(right_preds):,}")
print(f"\n DATA LOADING COMPLETE")

Loading data...

✓ Metadata loaded: Full (232), Left (116), Right (116)

✓ Summaries and Predictions loaded:
  Full: 46,400 | Left: 23,200 | Right: 23,200

 DATA LOADING COMPLETE


In [9]:
# =============================================================================
# CROSS-VALIDATION PERFORMANCE SUMMARY TABLE
# =============================================================================

def extract_cv_metrics(cv_summary, region_info):
    """Extract key metrics from CV summary"""
    
    # Get overall metrics
    overall = cv_summary['overall_metrics']
    n_samples = overall['n_samples']
    n_classes = overall['n_classes']
    
    # Mean validation accuracy (top-level)
    mean_acc = cv_summary['mean_val_accuracy']
    
    # Extract fold accuracies and calculate std, min, max
    fold_accs = [fold['val_accuracy'] for fold in cv_summary['fold_metrics']]
    std_acc = np.std(fold_accs)
    min_acc = min(fold_accs)
    max_acc = max(fold_accs)
    
    # Chance level (1 / number of classes)
    chance_level = 1 / n_classes
    
    # Fold above chance
    fold_above_chance = mean_acc / chance_level
    
    return {
        'n_regions': n_classes,  # n_classes corresponds to n_regions
        'n_samples': n_samples,
        'mean_acc': mean_acc,
        'std_acc': std_acc,
        'min_acc': min_acc,
        'max_acc': max_acc,
        'chance_level': chance_level,
        'fold_above_chance': fold_above_chance
    }

# Extract metrics for each model
full_metrics = extract_cv_metrics(full_cv, full_region_info)
left_metrics = extract_cv_metrics(left_cv, lh_region_info)
right_metrics = extract_cv_metrics(right_cv, rh_region_info)

# Create summary table
summary_data = {
    'Model': ['Full connectivity', 'Left hemisphere', 'Right hemisphere'],
    'N Regions': [full_metrics['n_regions'], left_metrics['n_regions'], right_metrics['n_regions']],
    'N Samples': [f"{full_metrics['n_samples']:,}", f"{left_metrics['n_samples']:,}", f"{right_metrics['n_samples']:,}"],
    'Mean CV Accuracy': [f"{full_metrics['mean_acc']:.1%}", f"{left_metrics['mean_acc']:.1%}", f"{right_metrics['mean_acc']:.1%}"],
    'Std Across Folds': [f"{full_metrics['std_acc']:.1%}", f"{left_metrics['std_acc']:.1%}", f"{right_metrics['std_acc']:.1%}"],
    'Range': [
        f"{full_metrics['min_acc']:.1%}-{full_metrics['max_acc']:.1%}",
        f"{left_metrics['min_acc']:.1%}-{left_metrics['max_acc']:.1%}",
        f"{right_metrics['min_acc']:.1%}-{right_metrics['max_acc']:.1%}"
    ],
    'Chance level': [f"{full_metrics['chance_level']:.2%}", f"{left_metrics['chance_level']:.2%}", f"{right_metrics['chance_level']:.2%}"],
    'Fold Above chance': [f"{full_metrics['fold_above_chance']:.0f}x", f"{left_metrics['fold_above_chance']:.0f}x", f"{right_metrics['fold_above_chance']:.0f}x"]
}

cv_summary_table = pd.DataFrame(summary_data)

# Display with nice formatting
print("\n" + "="*100)
print("CROSS-VALIDATION PERFORMANCE SUMMARY")
print("="*100 + "\n")
display(cv_summary_table)


CROSS-VALIDATION PERFORMANCE SUMMARY



,Model,N Regions,N Samples,Mean CV Accuracy,Std Across Folds,Range,Chance level,Fold Above chance
0,Full connectivity,232,"51,968",76.2%,0.7%,75.2%-77.4%,0.43%,177x
1,Left hemisphere,116,"25,984",92.8%,0.6%,91.8%-93.3%,0.86%,108x
2,Right hemisphere,116,"25,984",92.5%,0.3%,91.9%-92.9%,0.86%,107x


In [37]:
import os
os.getcwd()

'/home/sjoon/projects/brain_connectivity_classifier/analysis/JupyterFiles'